# ASD Preprocessing Dataset Builder

Refactored preprocessing notebook for generating the saved datasets used by the ASD training experiments.

## Shared preprocessing code

Run this cell once. The helper functions below read Original/Sheffield recordings, apply the requested preprocessing, window the data, and save each dataset folder.

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import os.path as op
import random
import re
import sys
import traceback
from pathlib import Path
from typing import Any, Optional, Sequence, Union

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pywt
from braindecode.datasets import BaseConcatDataset, BaseDataset
from braindecode.datautil import load_concat_dataset
from braindecode.preprocessing.windowers import create_windows_from_events
from mne.channels import make_standard_montage
from scipy.io import loadmat

try:
    from mne.preprocessing import ICA
except Exception:
    ICA = None

try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

try:
    from data_anonymization import anonymize_data, anonymize_dataset, deanonymize_data
except ImportError:
    def anonymize_data(value: Any) -> Any:
        return value

    def deanonymize_data(value: Any) -> Any:
        return value

    def anonymize_dataset(dataset: Any) -> Any:
        return dataset

try:
    sys.stdout.reconfigure(line_buffering=True)
except Exception:
    pass

mne.set_log_level("WARNING")

PathLike = Union[str, os.PathLike]

base_dir = "./"

SOURCE_DATA_DIRS: dict[str, str] = {
    "Original": op.join(base_dir, "ASD_data_copy"),
    "Sheffield": op.join(base_dir, "sheffield"),
}

# Keep these False for anonymized output.
STORE_REAL_FILENAMES_IN_DESCRIPTION = False
PRINT_REAL_FILENAMES = False


PRIVATE_FILENAME_CONFIG_PATH = op.join(base_dir, "private_filename_config.json")


def load_private_filename_config(path: PathLike = PRIVATE_FILENAME_CONFIG_PATH) -> dict[str, Any]:
    """Load private real-filename configuration from a local JSON file if it exists."""
    if not op.exists(path):
        return {"exclude_subjects": [], "special_crop_starts": {}}

    with open(path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    return {
        "exclude_subjects": list(cfg.get("exclude_subjects", [])),
        "special_crop_starts": dict(cfg.get("special_crop_starts", {})),
    }


_PRIVATE_FILENAME_CONFIG = load_private_filename_config()

EXCLUDE_SUBJECTS = set(_PRIVATE_FILENAME_CONFIG["exclude_subjects"])
SPECIAL_CROP_STARTS = dict(_PRIVATE_FILENAME_CONFIG["special_crop_starts"])

BACKGROUND_KEYS = [
    "Bgrnd",
    "Background",
    "Фоновая запись(testUser)",
    "Фоновий запис(testUser)(testUser)",
    "Фоновая запись (testUser)",
]

LEGACY_1020_MAP = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}
CASE_FIX_1020_MAP = {"FP1": "Fp1", "FP2": "Fp2", "FZ": "Fz", "CZ": "Cz", "PZ": "Pz"}

TARGET_CHANNELS = [
    "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8",
    "T7", "C3", "Cz", "C4", "T8",
    "P7", "P3", "Pz", "P4", "P8",
    "O1", "O2",
]

AUX_KEYS = ["ECG", "EKG", "EOG", "EMG", "PULSU", "CHASTOTA", "MA "]

PLOT_FIRST_N = 25
PLOT_DURATION_SEC = 150
PLOT_DECIM = 5
OUTPUT_SFREQ = 250
WINDOW_DURATION_SEC = 4
MIN_CHANNELS_FOR_INTERPOLATION = 3
MIN_CHANNELS_NO_INTERPOLATION = 1

LOW_CUT_HZ = 1.0
HIGH_CUT_HZ = 40.0
BUTTERWORTH_ORDER = 5

ICA_METHOD = "infomax"
ICA_RANDOM_STATE = 97
ICA_MAX_ITER = "auto"
ICA_N_COMPONENTS = 0.99
ICA_FIT_PARAMS = dict(extended=True)
ICA_FIT_L_FREQ = 1.0
ICA_FIT_H_FREQ = 100.0
ICA_KEEP_LABELS = {"brain", "other"}
ICA_EXCLUDE_THRESHOLD: Optional[float] = None

DWT_WAVELET = "db4"
DWT_LEVEL = 6
FREQUENCY_BANDS = ("alpha", "beta", "theta", "gamma", "delta")
DWT_COMBINED_BANDS = ("delta", "theta", "alpha", "beta", "gamma")
DWT_DETAIL_BY_BAND = {"gamma": 2, "beta": 3, "alpha": 4, "theta": 5, "delta": 6}
DWT_DELTA_INCLUDE_APPROXIMATION = True
DWT_COMBINED_RANDOM_STATE = 20260426

print("Preprocessing helpers loaded. Set RUN_DATASET_CREATION=True before running dataset cells.", flush=True)


def _stable_anonymous_id(value: Any, prefix: str = "record") -> str:
    """Return a stable anonymous identifier for a private filename/path."""
    text = str(value)
    digest = hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()[:10]
    return f"{prefix}_{digest}"


def _looks_like_private_filename_or_path(value: Any) -> bool:
    """Heuristic check for strings that look like real filenames or local paths."""
    text = str(value)
    lower = text.lower()

    if any(sep in text for sep in ["/", "\\"]):
        return True

    private_extensions = (
        ".edf", ".set", ".fdt", ".mat", ".vhdr", ".eeg", ".bdf", ".cnt",
        ".csv", ".xlsx", ".json", ".png", ".jpg", ".jpeg",
    )
    if lower.endswith(private_extensions):
        return True

    return False


def visible_name(value: Any, prefix: str = "record") -> str:
    """Return a safe display name that does not expose private filenames."""
    if PRINT_REAL_FILENAMES:
        return str(value)

    text = str(value)

    if _looks_like_private_filename_or_path(text):
        return _stable_anonymous_id(text, prefix=prefix)

    return str(anonymize_data(text))


def display_file_name(file_name: str = "") -> str:
    """Return a safe display name for a real filename."""
    return visible_name(file_name, prefix="file")


def sanitize_message(message: Any, hidden_values: Optional[Sequence[Any]] = None) -> str:
    """Remove known private filenames/paths from exception or log messages."""
    text = str(message)

    for value in hidden_values or []:
        if value is None:
            continue
        value_text = str(value)
        if value_text:
            text = text.replace(value_text, visible_name(value_text))

    # Hide common explicit path-like substrings that may appear in library exceptions.
    text = re.sub(r"[\w\-. /\\А-Яа-яІіЇїЄєҐґ]+?\.(edf|EDF|set|SET|fdt|FDT|mat|MAT)", "[hidden filename]", text)

    return text


def safe_stem(name: str) -> str:
    """Return a filesystem-safe anonymized stem for plots and temporary labels."""
    safe_name = visible_name(name)
    return Path(str(safe_name)).stem.replace(" ", "_").replace("/", "_").replace("\\", "_")


def normalize_name(ch: str) -> str:
    """Normalize raw EEG channel names before 10-20 mapping."""
    ch = str(ch).upper().strip().replace("EEG ", "")
    if "_" in ch:
        ch = ch.split("_")[0]
    return ch.replace("  ", " ")


def is_aux(ch: str) -> bool:
    """Return True when a channel looks like a non-EEG auxiliary channel."""
    return any(key in normalize_name(ch) for key in AUX_KEYS)


def detect_lead_system(ch_names: Sequence[str]) -> str:
    """Infer whether a recording is referential, average-referenced, or bipolar."""
    cleaned = [normalize_name(ch) for ch in ch_names]
    bipolar_count = 0

    for ch in cleaned:
        if "-" in ch:
            _, right = ch.split("-", 1)
            if right not in {"A1", "A2", "AV", "N"}:
                bipolar_count += 1

    if bipolar_count >= 5:
        return "bipolar"
    if any(ch.endswith("-AV") for ch in cleaned):
        return "average_reference"
    if any(ch.endswith("-A1") for ch in cleaned) and any(ch.endswith("-A2") for ch in cleaned):
        return "mixed_a1_a2_reference"
    if any(ch.endswith("-A1") for ch in cleaned):
        return "a1_reference"
    if any(ch.endswith("-A2") for ch in cleaned):
        return "a2_reference"

    return "unknown"


def sensor_name(ch: str) -> Optional[str]:
    """Map a raw channel name to its canonical 10-20 sensor name."""
    ch = normalize_name(ch)

    if "-" in ch:
        left, right = ch.split("-", 1)
        if right in {"A1", "A2", "AV", "N"}:
            ch = left
        else:
            return None

    ch = LEGACY_1020_MAP.get(ch, ch)
    return CASE_FIX_1020_MAP.get(ch, ch)


def public_record_name(record: dict[str, Any]) -> str:
    """Return a stable non-identifying name for prints, plots, and descriptions."""
    if PRINT_REAL_FILENAMES:
        return str(record.get("file", record.get("public_id", "record")))

    return str(anonymize_data(record.get("public_id", "record")))


def get_crop_start(file_name: str) -> float:
    """Return the manually configured crop start for a recording, if any."""
    return float(SPECIAL_CROP_STARTS.get(file_name, 0.0))


def crop_raw_to_duration(
    raw: mne.io.BaseRaw,
    file_name: str = "",
    duration_sec: float = PLOT_DURATION_SEC,
) -> mne.io.BaseRaw:
    """Copy and crop a raw recording to the configured analysis duration."""
    cropped = raw.copy()
    start_t = get_crop_start(file_name)

    if start_t >= cropped.times[-1]:
        raise ValueError("Configured crop start is outside recording duration")

    max_t = min(start_t + float(duration_sec), cropped.times[-1])
    cropped.crop(tmin=start_t, tmax=max_t)

    return cropped


def rename_and_drop_channels(raw: mne.io.BaseRaw) -> mne.io.BaseRaw:
    """Drop non-target channels and rename usable EEG channels to 10-20 names."""
    rename_map: dict[str, str] = {}
    drop_chs: list[str] = []

    for ch in list(raw.ch_names):
        new_name = None if is_aux(ch) else sensor_name(ch)

        if new_name is None or new_name not in TARGET_CHANNELS:
            drop_chs.append(ch)
        else:
            rename_map[ch] = new_name

    if drop_chs:
        raw.drop_channels([ch for ch in drop_chs if ch in raw.ch_names], on_missing="ignore")

    used: set[str] = set()
    safe_map: dict[str, str] = {}
    extra_drop: list[str] = []

    for old, new in rename_map.items():
        if old not in raw.ch_names:
            continue

        if new in used:
            extra_drop.append(old)
        else:
            safe_map[old] = new
            used.add(new)

    if extra_drop:
        raw.drop_channels([ch for ch in extra_drop if ch in raw.ch_names], on_missing="ignore")

    if safe_map:
        raw.rename_channels(safe_map)

    return raw


def prepare_no_interpolation_channels(raw: mne.io.BaseRaw) -> mne.io.BaseRaw:
    """Keep usable non-auxiliary EEG channels without forcing TARGET_CHANNELS."""
    raw = raw.copy()

    drop_chs = [ch for ch in list(raw.ch_names) if is_aux(ch)]
    if drop_chs:
        raw.drop_channels([ch for ch in drop_chs if ch in raw.ch_names], on_missing="ignore")
        print(f"  dropped auxiliary channels for no-interpolation: {drop_chs}", flush=True)

    rename_map: dict[str, str] = {}
    used_new_names: set[str] = set(raw.ch_names)

    for ch in list(raw.ch_names):
        new_name = sensor_name(ch)

        if new_name is None:
            continue
        if new_name == ch:
            continue
        if new_name in used_new_names:
            continue

        rename_map[ch] = new_name
        used_new_names.add(new_name)

    if rename_map:
        raw.rename_channels(rename_map)
        print(f"  renamed no-interpolation channels: {rename_map}", flush=True)

    return raw


def drop_empty_channels(raw: mne.io.BaseRaw, min_std: float = 1e-12) -> list[str]:
    """Drop channels with non-finite values or effectively empty signal."""
    bad: list[str] = []

    for ch in list(raw.ch_names):
        data = raw.get_data(picks=[ch])[0]

        if data.size == 0 or not np.isfinite(data).all() or float(np.nanstd(data)) < min_std:
            bad.append(ch)

    if bad:
        raw.drop_channels(bad, on_missing="ignore")
        print(f"  dropped empty/non-finite channels: {bad}", flush=True)

    return bad


def target_montage() -> mne.channels.DigMontage:
    """Build a 10-20 montage containing exactly TARGET_CHANNELS."""
    montage_full = make_standard_montage("standard_1020")
    pos_full = montage_full.get_positions()["ch_pos"]
    ch_pos = {ch: pos_full[ch] for ch in TARGET_CHANNELS if ch in pos_full}

    return mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame="head")


def unify_raw_to_1020(
    raw: mne.io.BaseRaw,
    file_name: str = "",
    apply_interpolation: bool = True,
    crop_150s: bool = True,
    drop_empty: bool = True,
    require_full_target_set: bool = False,
) -> mne.io.BaseRaw:
    """Crop, clean channels, optionally interpolate to TARGET_CHANNELS, and resample."""
    print(f"  unify start: {display_file_name(file_name)}", flush=True)

    system = detect_lead_system(raw.ch_names)
    print(f"  detected system: {system}", flush=True)

    if system == "bipolar" and apply_interpolation:
        raise ValueError("Bipolar recording cannot be safely converted to common referential montage")

    if system == "bipolar" and not apply_interpolation:
        print("  bipolar/no-interpolation: keeping original non-auxiliary channel layout", flush=True)

    out = raw.copy()

    if crop_150s:
        out = crop_raw_to_duration(out, file_name=file_name, duration_sec=PLOT_DURATION_SEC)

    out.load_data()

    if apply_interpolation:
        out = rename_and_drop_channels(out)

        if drop_empty:
            drop_empty_channels(out)

        present = [ch for ch in TARGET_CHANNELS if ch in out.ch_names]
        print(f"  present target channels ({len(present)}): {present}", flush=True)

        if len(present) < MIN_CHANNELS_FOR_INTERPOLATION:
            raise ValueError(f"Too few usable EEG channels for interpolation: {present}")

        out.pick(present)
        out.set_montage("standard_1020", match_case=False, on_missing="ignore")

        montage = out.get_montage()
        if montage is None:
            raise ValueError("Montage was not set successfully")

        pos = montage.get_positions()["ch_pos"]
        valid_present = [
            ch for ch in out.ch_names
            if ch in pos and np.isfinite(pos[ch]).all()
        ]

        if len(valid_present) < MIN_CHANNELS_FOR_INTERPOLATION:
            raise ValueError(f"Too few channels with valid montage positions: {valid_present}")

        out.pick(valid_present)

        print("  before interpolate_to", flush=True)
        out = out.interpolate_to(target_montage(), method="spline")
        out.reorder_channels([ch for ch in TARGET_CHANNELS if ch in out.ch_names])
        print("  after interpolate_to", flush=True)

    else:
        out = prepare_no_interpolation_channels(out)

        if drop_empty:
            drop_empty_channels(out)

        if len(out.ch_names) < MIN_CHANNELS_NO_INTERPOLATION:
            raise ValueError("No usable EEG channels remain after dropping auxiliary/empty channels")

        print(f"  no-interpolation usable channels ({len(out.ch_names)}): {out.ch_names}", flush=True)

    out.resample(OUTPUT_SFREQ, verbose=False)
    print("  unify done", flush=True)

    return out


def crop_to_plot_duration(raw: mne.io.BaseRaw, duration_sec: float = PLOT_DURATION_SEC) -> mne.io.BaseRaw:
    """Copy and crop a raw object for compact diagnostic plots."""
    out = raw.copy()
    out.crop(tmin=0.0, tmax=min(float(duration_sec), out.times[-1]))
    return out


def make_before_plotable(raw_before: mne.io.BaseRaw) -> mne.io.BaseRaw:
    """Prepare a preprocessed-before signal for electrode and signal plots."""
    raw = raw_before.copy().load_data()
    raw = rename_and_drop_channels(raw)
    drop_empty_channels(raw)

    keep = [ch for ch in TARGET_CHANNELS if ch in raw.ch_names]
    raw.pick(keep)
    raw.set_montage("standard_1020", match_case=False, on_missing="ignore")

    return raw


def plot_signals_before_after(
    raw_before: mne.io.BaseRaw,
    raw_after: mne.io.BaseRaw,
    file_name: str = "",
    duration_sec: float = PLOT_DURATION_SEC,
    decim: int = PLOT_DECIM,
    save_path: Optional[PathLike] = None,
) -> None:
    """Plot raw signal traces before and after preprocessing."""
    file_name = visible_name(file_name)

    raw_before = crop_to_plot_duration(raw_before, duration_sec)
    raw_after = crop_to_plot_duration(raw_after, duration_sec)

    data_before = raw_before.get_data()[:, ::decim]
    data_after = raw_after.get_data()[:, ::decim]

    times_before = np.arange(data_before.shape[1]) / (raw_before.info["sfreq"] / decim)
    times_after = np.arange(data_after.shape[1]) / (raw_after.info["sfreq"] / decim)

    fig, axes = plt.subplots(2, 1, figsize=(22, 12), sharex=False)

    scale_before = float(np.nanstd(data_before) * 5) if np.nanstd(data_before) > 0 else 1e-5
    for i in range(data_before.shape[0]):
        axes[0].plot(times_before, data_before[i] + i * scale_before, linewidth=0.5)

    axes[0].set_title(f"{file_name} - BEFORE processing")
    axes[0].set_ylabel("Channels (offset)")
    axes[0].set_yticks([i * scale_before for i in range(data_before.shape[0])])
    axes[0].set_yticklabels(raw_before.ch_names, fontsize=8)

    scale_after = float(np.nanstd(data_after) * 5) if np.nanstd(data_after) > 0 else 1e-5
    for i in range(data_after.shape[0]):
        axes[1].plot(times_after, data_after[i] + i * scale_after, linewidth=0.5)

    axes[1].set_title(f"{file_name} - AFTER processing")
    axes[1].set_ylabel("Channels (offset)")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_yticks([i * scale_after for i in range(data_after.shape[0])])
    axes[1].set_yticklabels(raw_after.ch_names, fontsize=8)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


def plot_electrode_maps_before_after(
    raw_before: mne.io.BaseRaw,
    raw_after: mne.io.BaseRaw,
    file_name: str = "",
    save_path: Optional[PathLike] = None,
) -> None:
    """Plot electrode layouts before and after preprocessing."""
    file_name = visible_name(file_name)

    raw_before_plot = make_before_plotable(raw_before)
    raw_after_plot = crop_to_plot_duration(raw_after.copy())

    fig = plt.figure(figsize=(14, 6))

    ax1 = fig.add_subplot(1, 2, 1)
    mne.viz.plot_sensors(raw_before_plot.info, kind="topomap", show_names=True, axes=ax1, show=False)
    ax1.set_title(f"{file_name} - BEFORE electrodes")

    ax2 = fig.add_subplot(1, 2, 2)
    mne.viz.plot_sensors(raw_after_plot.info, kind="topomap", show_names=True, axes=ax2, show=False)
    ax2.set_title(f"{file_name} - AFTER electrodes")

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


def plot_psd_before_after(
    raw_before: mne.io.BaseRaw,
    raw_after: mne.io.BaseRaw,
    file_name: str = "",
    fmin: float = 1.0,
    fmax: float = 40.0,
    save_path: Optional[PathLike] = None,
) -> None:
    """Plot PSD before and after preprocessing."""
    file_name = visible_name(file_name)

    raw_before = crop_to_plot_duration(raw_before)
    raw_after = crop_to_plot_duration(raw_after)

    psd_before = raw_before.compute_psd(fmin=fmin, fmax=fmax, verbose=False)
    psd_after = raw_after.compute_psd(fmin=fmin, fmax=fmax, verbose=False)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    psd_before.plot(axes=axes[0], show=False)
    axes[0].set_title(f"{file_name} - BEFORE PSD")

    psd_after.plot(axes=axes[1], show=False)
    axes[1].set_title(f"{file_name} - AFTER PSD")

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


def apply_butterworth_filter(
    raw: mne.io.BaseRaw,
    l_freq: float = LOW_CUT_HZ,
    h_freq: float = HIGH_CUT_HZ,
    order: int = BUTTERWORTH_ORDER,
) -> mne.io.BaseRaw:
    """Apply an IIR Butterworth band-pass filter to a unified raw signal."""
    raw_filt = raw.copy().load_data()
    raw_filt.filter(
        l_freq=l_freq,
        h_freq=h_freq,
        method="iir",
        iir_params={"order": order, "ftype": "butter"},
        verbose=False,
    )
    return raw_filt


def fit_label_and_apply_ica(
    raw: mne.io.BaseRaw,
    file_name: str = "",
    save_plots: bool = False,
    ica_plots_dir: Optional[PathLike] = None,
) -> tuple[mne.io.BaseRaw, dict[str, Any]]:
    """Fit ICA, label components with ICLabel, and remove non-brain components."""
    if ICA is None or label_components is None:
        raise ImportError("ICA preprocessing requires mne.preprocessing.ICA and mne_icalabel.label_components")

    shown_file_name = visible_name(file_name)

    raw_fit = raw.copy().load_data()
    raw_fit.set_eeg_reference("average", projection=False, verbose=False)
    raw_fit.filter(l_freq=ICA_FIT_L_FREQ, h_freq=ICA_FIT_H_FREQ, picks="eeg", verbose=False)

    ica = ICA(
        n_components=ICA_N_COMPONENTS,
        method=ICA_METHOD,
        max_iter=ICA_MAX_ITER,
        random_state=ICA_RANDOM_STATE,
        fit_params=ICA_FIT_PARAMS,
    )

    ica.fit(raw_fit, picks="eeg")

    ic_labels = label_components(raw_fit, ica, method="iclabel")
    labels = list(ic_labels["labels"])
    y_pred_proba = ic_labels.get("y_pred_proba", None)

    exclude_idx: list[int] = []

    for idx, label in enumerate(labels):
        if label in ICA_KEEP_LABELS:
            continue

        if ICA_EXCLUDE_THRESHOLD is None:
            exclude_idx.append(idx)
            continue

        prob = None
        if y_pred_proba is not None:
            try:
                prob = float(np.max(y_pred_proba[idx]))
            except Exception:
                prob = None

        if prob is None or prob >= ICA_EXCLUDE_THRESHOLD:
            exclude_idx.append(idx)

    print(f"  ICA labels for {shown_file_name}:", flush=True)

    for idx, label in enumerate(labels):
        prob_txt = ""

        if y_pred_proba is not None:
            try:
                prob_txt = f", max_prob={float(np.max(y_pred_proba[idx])):.4f}"
            except Exception:
                prob_txt = ""

        mark = " <-- EXCLUDE" if idx in exclude_idx else ""
        print(f"    ICA{idx:02d}: {label}{prob_txt}{mark}", flush=True)

    print(f"  ICA components to exclude for {shown_file_name}: {exclude_idx}", flush=True)

    raw_clean = raw.copy().load_data()
    ica.apply(raw_clean, exclude=exclude_idx)

    if save_plots and ica_plots_dir is not None:
        os.makedirs(ica_plots_dir, exist_ok=True)

        stem = safe_stem(file_name)

        fig = ica.plot_components(show=False)
        figs = fig if isinstance(fig, list) else [fig]

        for j, one_fig in enumerate(figs):
            one_fig.savefig(
                op.join(ica_plots_dir, f"{stem}_ica_components_{j:02d}.png"),
                dpi=200,
                bbox_inches="tight",
            )
            plt.close(one_fig)

        fig = ica.plot_sources(raw_fit, show=False)
        fig.savefig(op.join(ica_plots_dir, f"{stem}_ica_sources.png"), dpi=200, bbox_inches="tight")
        plt.close(fig)

        fig = ica.plot_overlay(raw_fit, exclude=exclude_idx, picks="eeg", show=False)
        fig.savefig(op.join(ica_plots_dir, f"{stem}_ica_overlay.png"), dpi=200, bbox_inches="tight")
        plt.close(fig)

    info = {
        "labels": labels,
        "exclude_idx": exclude_idx,
        "y_pred_proba": y_pred_proba,
        "n_components_": getattr(ica, "n_components_", None),
        "n_iter_": getattr(ica, "n_iter_", None),
    }

    return raw_clean, info


def reconstruct_dwt_band_1d(
    signal: np.ndarray,
    band: str,
    wavelet: str = DWT_WAVELET,
    level: int = DWT_LEVEL,
) -> np.ndarray:
    """Reconstruct one EEG frequency band from a 1D signal using DWT coefficients."""
    band = band.lower()

    if band not in DWT_DETAIL_BY_BAND:
        raise ValueError(f"Unsupported DWT band: {band}. Use one of {sorted(DWT_DETAIL_BY_BAND)}")

    signal = np.asarray(signal, dtype=np.float64)
    wavelet_obj = pywt.Wavelet(wavelet)
    max_level = pywt.dwt_max_level(len(signal), wavelet_obj.dec_len)

    if max_level < 1:
        return signal.copy()

    level = min(level, max_level)
    coeffs = pywt.wavedec(signal, wavelet=wavelet, level=level)
    keep = [np.zeros_like(c) for c in coeffs]

    if band == "delta" and DWT_DELTA_INCLUDE_APPROXIMATION:
        keep[0] = coeffs[0]

    target_detail = DWT_DETAIL_BY_BAND[band]
    coeff_index = level - target_detail + 1

    if 1 <= coeff_index < len(coeffs):
        keep[coeff_index] = coeffs[coeff_index]

    reconstructed = pywt.waverec(keep, wavelet=wavelet)

    return reconstructed[: len(signal)]


def reconstruct_dwt_bands_1d(
    signal: np.ndarray,
    bands: Sequence[str] = DWT_COMBINED_BANDS,
    wavelet: str = DWT_WAVELET,
    level: int = DWT_LEVEL,
) -> np.ndarray:
    """Reconstruct a signal from several selected DWT frequency bands."""
    signal = np.asarray(signal, dtype=np.float64)
    wavelet_obj = pywt.Wavelet(wavelet)
    max_level = pywt.dwt_max_level(len(signal), wavelet_obj.dec_len)

    if max_level < 1:
        return signal.copy()

    level = min(level, max_level)
    coeffs = pywt.wavedec(signal, wavelet=wavelet, level=level)
    keep = [np.zeros_like(c) for c in coeffs]

    for band in bands:
        band = band.lower()

        if band == "delta" and DWT_DELTA_INCLUDE_APPROXIMATION:
            keep[0] = coeffs[0]

        target_detail = DWT_DETAIL_BY_BAND[band]
        coeff_index = level - target_detail + 1

        if 1 <= coeff_index < len(coeffs):
            keep[coeff_index] = coeffs[coeff_index]

    reconstructed = pywt.waverec(keep, wavelet=wavelet)

    return reconstructed[: len(signal)]


def reconstruct_dwt_band_raw(
    raw: mne.io.BaseRaw,
    band: str,
    wavelet: str = DWT_WAVELET,
    level: int = DWT_LEVEL,
) -> mne.io.BaseRaw:
    """Apply single-band DWT reconstruction to every channel in a raw signal."""
    raw_band = raw.copy().load_data()
    data = raw_band.get_data()
    reconstructed = np.zeros_like(data, dtype=np.float64)

    for ch_i in range(data.shape[0]):
        reconstructed[ch_i] = reconstruct_dwt_band_1d(
            data[ch_i],
            band=band,
            wavelet=wavelet,
            level=level,
        )

    raw_band._data = reconstructed

    return raw_band


def reconstruct_dwt_bands_raw(
    raw: mne.io.BaseRaw,
    bands: Sequence[str] = DWT_COMBINED_BANDS,
) -> mne.io.BaseRaw:
    """Apply multi-band DWT reconstruction to every channel in a raw signal."""
    raw_bands = raw.copy().load_data()
    data = raw_bands.get_data()
    reconstructed = np.zeros_like(data, dtype=np.float64)

    for ch_i in range(data.shape[0]):
        reconstructed[ch_i] = reconstruct_dwt_bands_1d(data[ch_i], bands=bands)

    raw_bands._data = reconstructed

    return raw_bands


def create_dwt_combined_raw(
    template_raw: mne.io.BaseRaw,
    donor_raws_by_band: dict[str, mne.io.BaseRaw],
    bands: Sequence[str] = DWT_COMBINED_BANDS,
) -> mne.io.BaseRaw:
    """Create one artificial raw by combining DWT bands drawn from donor recordings."""
    template = template_raw.copy().load_data()
    min_n_times = min([template.n_times] + [donor.n_times for donor in donor_raws_by_band.values()])
    combined = np.zeros((len(template.ch_names), min_n_times), dtype=np.float64)

    for band in bands:
        donor = donor_raws_by_band[band].copy().load_data()
        donor_data = donor.get_data()[:, :min_n_times]

        for ch_i in range(donor_data.shape[0]):
            combined[ch_i] += reconstruct_dwt_band_1d(donor_data[ch_i], band=band)

    if template.n_times != min_n_times:
        template.crop(tmin=0.0, tmax=(min_n_times - 1) / float(template.info["sfreq"]))

    template._data = combined

    return template


def load_setname(file_path: PathLike) -> str:
    """Read the EEG.setname field from a Sheffield EEGLAB .set file."""
    eeg = loadmat(file_path, squeeze_me=True, struct_as_record=False).get("EEG")
    return str(getattr(eeg, "setname", "")).strip()


def infer_label(setname: str) -> str:
    """Infer the Sheffield label from the EEGLAB set name."""
    if setname.startswith("ASD"):
        return "autism"
    if setname.startswith("P"):
        return "norm"

    raise ValueError(f"Unknown setname prefix for label inference: {setname}")


def sheffield_sort_key(name: str) -> tuple[int, str]:
    """Sort Sheffield files by leading number when available."""
    match = re.match(r"(\d+)", name)
    return (int(match.group(1)) if match else 10**9, name)


def read_original_records(
    data_dir: PathLike,
    exclude_subjects: set[str] = EXCLUDE_SUBJECTS,
) -> list[dict[str, Any]]:
    """Read Original EDF metadata and lazy MNE Raw handles."""
    records: list[dict[str, Any]] = []
    subject_id = 1

    for label, subdir in [("autism", "Autism"), ("norm", "Norm")]:
        dir_path = op.join(data_dir, subdir)

        if not op.isdir(dir_path):
            print(f"Missing Original subfolder: {visible_name(dir_path)}", flush=True)
            continue

        for file in sorted(os.listdir(dir_path)):
            if not file.lower().endswith(".edf"):
                continue

            public_id = f"original_{subject_id:03d}_{label}"

            if file in exclude_subjects:
                print(f"SKIP {public_id}: excluded", flush=True)
                continue

            file_path = op.join(dir_path, file)

            try:
                print(f"Reading {public_id}", flush=True)

                raw = mne.io.read_raw_edf(
                    file_path,
                    stim_channel="auto",
                    preload=False,
                    verbose=False,
                )

                _, event_dict = mne.events_from_annotations(raw, verbose=False)

                records.append({
                    "origin": "Original",
                    "file": file,
                    "filepath": file_path,
                    "public_id": public_id,
                    "subject_id": subject_id,
                    "raw": raw,
                    "event_dict": event_dict,
                    "label": label,
                })

                print(f"OK read {public_id}", flush=True)
                subject_id += 1

            except Exception as exc:
                msg = sanitize_message(exc, hidden_values=[file, file_path, dir_path])
                print(f"FAIL reading {public_id}: {type(exc).__name__}: {msg}", flush=True)

                if PRINT_REAL_FILENAMES:
                    traceback.print_exc()

    return records


def read_sheffield_records(data_dir: PathLike) -> list[dict[str, Any]]:
    """Read Sheffield EEGLAB metadata and lazy MNE Raw handles."""
    records: list[dict[str, Any]] = []

    if not op.isdir(data_dir):
        print(f"Missing Sheffield folder: {visible_name(data_dir)}", flush=True)
        return records

    set_files = sorted(
        [f for f in os.listdir(data_dir) if f.lower().endswith(".set")],
        key=sheffield_sort_key,
    )

    for subject_id, file in enumerate(set_files, start=1):
        file_path = op.join(data_dir, file)
        public_id = f"sheffield_{subject_id:03d}"

        try:
            setname = load_setname(file_path)
            label = infer_label(setname)

            raw = mne.io.read_raw_eeglab(
                file_path,
                preload=False,
                verbose=False,
            )

            records.append({
                "origin": "Sheffield",
                "file": file,
                "filepath": file_path,
                "public_id": f"{public_id}_{label}",
                "subject_id": subject_id,
                "setname": setname,
                "label": label,
                "raw": raw,
            })

            print(f"OK read {public_id}_{label}", flush=True)

        except Exception as exc:
            msg = sanitize_message(exc, hidden_values=[file, file_path, data_dir])
            print(f"FAIL reading {public_id}: {type(exc).__name__}: {msg}", flush=True)

            if PRINT_REAL_FILENAMES:
                traceback.print_exc()

    return records


def record_has_background(record: dict[str, Any]) -> bool:
    """Return True when an Original recording contains a background annotation."""
    event_keys = record.get("event_dict", {}).keys()
    return any(key in event_keys for key in BACKGROUND_KEYS)


def normalize_origin(origin: str) -> str:
    """Return a canonical dataset origin name."""
    aliases = {
        "original": "Original",
        "old": "Original",
        "sheffield": "Sheffield",
        "mixed": "Mixed",
    }

    key = str(origin).strip().lower()

    if key not in aliases:
        raise ValueError("Unsupported origin. Use Original, Sheffield, or Mixed.")

    return aliases[key]


def load_records(origin: str) -> list[dict[str, Any]]:
    """Load record descriptors for one raw dataset origin."""
    origin = normalize_origin(origin)
    data_dir = SOURCE_DATA_DIRS[origin]

    print(f"Loading {origin} records", flush=True)

    if origin == "Original":
        return read_original_records(data_dir)

    if origin == "Sheffield":
        return read_sheffield_records(data_dir)

    raise ValueError("Mixed records are created from already saved Original and Sheffield datasets")


def normalize_filter_type(filter_type: Optional[str]) -> str:
    """Return a canonical preprocessing filter name."""
    if filter_type is None:
        return "None"

    key = str(filter_type).strip().lower()

    aliases = {
        "": "None",
        "none": "None",
        "no filtering": "None",
        "no_filter": "None",
        "unfiltered": "None",
        "butter": "Butterworth",
        "butterworth": "Butterworth",
        "ica": "ICA",
        "dwt": "DWT",
        "dwt_combined": "DWT_COMBINED",
        "dwt combined": "DWT_COMBINED",
        "combined_dwt": "DWT_COMBINED",
    }

    if key not in aliases:
        raise ValueError("Unsupported filter_type. Use None, Butterworth, ICA, DWT, or DWT_COMBINED.")

    return aliases[key]


def output_name_for(
    origin: str,
    filter_type: Optional[str],
    band: Optional[str] = None,
    apply_interpolation: bool = True,
) -> str:
    """Return the dataset folder suffix for a preprocessing combination."""
    origin = normalize_origin(origin)
    filter_name = normalize_filter_type(filter_type)
    band_name = band.lower() if isinstance(band, str) else None

    if not apply_interpolation:
        if origin == "Mixed":
            raise ValueError("Mixed no-interpolation dataset is intentionally unsupported.")
        return "150_no_interpolation" if origin == "Original" else "150_sheffield_no_interpolation"

    if filter_name == "None":
        return {
            "Original": "150",
            "Sheffield": "150_sheffield",
            "Mixed": "mixed_150",
        }[origin]

    if filter_name == "Butterworth":
        return {
            "Original": "butterworth",
            "Sheffield": "butterworth_sheffield",
            "Mixed": "mixed_butterworth",
        }[origin]

    if filter_name == "ICA":
        return {
            "Original": "original_ica",
            "Sheffield": "sheffield_ica",
            "Mixed": "mixed_ica",
        }[origin]

    if filter_name == "DWT":
        if band_name is None or band_name not in FREQUENCY_BANDS:
            raise ValueError(f"DWT requires band in {FREQUENCY_BANDS}")

        if origin == "Original":
            return f"{band_name}_dwt"

        if origin == "Sheffield":
            return f"{band_name}_dwt_sheffield"

        return f"mixed_{band_name}_dwt"

    if filter_name == "DWT_COMBINED":
        return {
            "Original": "dwt_combined",
            "Sheffield": "dwt_combined_sheffield",
            "Mixed": "mixed_dwt_combined",
        }[origin]

    raise ValueError(f"Unsupported filter type: {filter_type}")


def dataset_paths(output_name: str) -> dict[str, str]:
    """Return base, windows, plot, and ICA-plot paths for an output name."""
    return {
        "dataset": op.join(base_dir, f"dataset_{output_name}"),
        "windows": op.join(base_dir, f"windows_dataset_{output_name}"),
        "plots": op.join(base_dir, f"plots_before_after_{output_name}"),
        "ica_plots": op.join(base_dir, f"plots_ica_{output_name}"),
    }


def apply_requested_filter(
    raw: mne.io.BaseRaw,
    filter_name: str,
    band: Optional[str] = None,
    file_name: str = "",
    save_plots: bool = False,
    ica_plots_dir: Optional[PathLike] = None,
) -> tuple[mne.io.BaseRaw, dict[str, Any]]:
    """Apply the selected filter to an already unified raw signal."""
    if filter_name == "None" or filter_name == "DWT_COMBINED":
        return raw, {}

    if filter_name == "Butterworth":
        return apply_butterworth_filter(raw), {}

    if filter_name == "ICA":
        return fit_label_and_apply_ica(
            raw,
            file_name=file_name,
            save_plots=save_plots,
            ica_plots_dir=ica_plots_dir,
        )

    if filter_name == "DWT":
        if band is None:
            raise ValueError("Single-band DWT preprocessing requires band.")

        return reconstruct_dwt_band_raw(raw, band=band), {"dwt_band": band}

    raise ValueError(f"Unsupported filter: {filter_name}")


def set_background_annotation(raw: mne.io.BaseRaw, label: str) -> None:
    """Annotate the full processed recording with the Braindecode event label."""
    desc = "Bgrnd_autism" if label == "autism" else "Bgrnd"
    duration = raw.n_times / float(raw.info["sfreq"])
    raw.set_annotations(mne.Annotations(onset=[0.0], duration=[duration], description=[desc]))


def make_description(
    record: dict[str, Any],
    raw_after: mne.io.BaseRaw,
    filter_name: str,
    band: Optional[str],
    extra: Optional[dict[str, Any]] = None,
) -> dict[str, Any]:
    """Create a Braindecode description row without exposing real filenames by default."""
    desc: dict[str, Any] = {
        "event_code": 0 if record["label"] == "norm" else 1,
        "subject": record["subject_id"],
        "file": public_record_name(record),
        "label": record["label"],
        "origin": record["origin"],
        "filter_type": filter_name,
        "frequency_band": band or "",
        "lead_system_before": detect_lead_system(record["raw"].info["ch_names"]),
        "n_channels_after": len(raw_after.ch_names),
        "channels_after": ",".join(raw_after.ch_names),
    }

    if STORE_REAL_FILENAMES_IN_DESCRIPTION:
        desc["source_file"] = record.get("file", "")
        desc["filepath"] = record.get("filepath", "")

    if extra:
        desc.update(extra)

    return desc


def save_diagnostic_plots(
    raw_before: mne.io.BaseRaw,
    raw_after: mne.io.BaseRaw,
    display_name: str,
    plots_dir: PathLike,
) -> None:
    """Save before/after signal, montage, and PSD diagnostics for one recording."""
    os.makedirs(plots_dir, exist_ok=True)

    safe_display_name = visible_name(display_name)
    stem = safe_stem(safe_display_name)

    before_plot = make_before_plotable(raw_before)

    plot_signals_before_after(
        before_plot,
        raw_after,
        file_name=safe_display_name,
        save_path=op.join(plots_dir, f"{stem}_signals_before_after.png"),
    )

    plot_electrode_maps_before_after(
        raw_before,
        raw_after,
        file_name=safe_display_name,
        save_path=op.join(plots_dir, f"{stem}_electrodes_before_after.png"),
    )

    plot_psd_before_after(
        before_plot,
        raw_after,
        file_name=safe_display_name,
        save_path=op.join(plots_dir, f"{stem}_psd_before_after.png"),
    )


def keep_most_common_channel_count(
    parts: Sequence[mne.io.BaseRaw],
    descriptions: Sequence[dict[str, Any]],
    dataset_label: str = "dataset",
) -> tuple[list[mne.io.BaseRaw], list[dict[str, Any]]]:
    """Keep only recordings with the most frequent channel count."""
    if not parts:
        return [], []

    channel_counts = [len(raw.ch_names) for raw in parts]
    counts = pd.Series(channel_counts).value_counts().sort_index()
    mode_count = int(pd.Series(channel_counts).mode().iloc[0])

    print(f"\nNo-interpolation channel-count distribution for {dataset_label}:", flush=True)

    for n_channels, n_recordings in counts.items():
        print(f"  {int(n_channels)} channels: {int(n_recordings)} recordings", flush=True)

    print(f"Keeping only {dataset_label} recordings with the most common channel count: {mode_count}", flush=True)

    kept_parts: list[mne.io.BaseRaw] = []
    kept_descriptions: list[dict[str, Any]] = []
    skipped = 0

    for raw, desc in zip(parts, descriptions):
        n_channels = len(raw.ch_names)

        if n_channels == mode_count:
            new_desc = dict(desc)
            new_desc["no_interpolation_channel_count_mode"] = mode_count
            new_desc["no_interpolation_kept_by_channel_count"] = True
            kept_parts.append(raw)
            kept_descriptions.append(new_desc)
        else:
            skipped += 1
            print(
                f"  SKIP {desc.get('file', '[hidden filename]')}: "
                f"{n_channels} channels != mode {mode_count}",
                flush=True,
            )

    print(
        f"Kept {len(kept_parts)} recordings and skipped {skipped} recordings "
        f"after channel-count mode filtering.",
        flush=True,
    )

    if not kept_parts:
        raise RuntimeError("No recordings remain after filtering by most common channel count.")

    return kept_parts, kept_descriptions


def preprocess_records_to_parts(
    records: Sequence[dict[str, Any]],
    filter_name: str,
    band: Optional[str],
    apply_interpolation: bool,
    plots_dir: PathLike,
    ica_plots_dir: Optional[PathLike] = None,
    save_plots: bool = True,
    plot_first_n: int = PLOT_FIRST_N,
) -> tuple[list[mne.io.BaseRaw], list[dict[str, Any]]]:
    """Convert raw record descriptors into processed Raw parts and descriptions."""
    parts: list[mne.io.BaseRaw] = []
    descriptions: list[dict[str, Any]] = []
    plot_counter = 0

    for i, record in enumerate(records, start=1):
        display_name = public_record_name(record)
        real_file_name = record["file"]
        raw_before: Optional[mne.io.BaseRaw] = None

        try:
            print(f"[{i}/{len(records)}] START {display_name}", flush=True)

            raw_before = crop_raw_to_duration(
                record["raw"],
                file_name=real_file_name,
            ).load_data()

            raw_base = unify_raw_to_1020(
                record["raw"],
                file_name=real_file_name,
                apply_interpolation=apply_interpolation,
                crop_150s=True,
                drop_empty=True,
                require_full_target_set=False,
            )

            raw_after, filter_info = apply_requested_filter(
                raw_base,
                filter_name=filter_name,
                band=band,
                file_name=display_name,
                save_plots=save_plots and plot_counter < plot_first_n,
                ica_plots_dir=ica_plots_dir,
            )

            set_background_annotation(raw_after, record["label"])

            if save_plots and plot_counter < plot_first_n:
                print(f"[{i}/{len(records)}] saving diagnostic plots: {display_name}", flush=True)

                try:
                    save_diagnostic_plots(
                        raw_before,
                        raw_after,
                        display_name=display_name,
                        plots_dir=plots_dir,
                    )
                    plot_counter += 1

                except Exception as plot_exc:
                    msg = sanitize_message(
                        plot_exc,
                        hidden_values=[real_file_name, record.get("filepath", "")],
                    )
                    print(
                        f"[{i}/{len(records)}] diagnostic plots skipped for "
                        f"{display_name}: {type(plot_exc).__name__}: {msg}",
                        flush=True,
                    )

                    if PRINT_REAL_FILENAMES:
                        traceback.print_exc()

            descriptions.append(make_description(record, raw_after, filter_name, band, filter_info))
            parts.append(raw_after)

            print(f"[{i}/{len(records)}] ADDED {display_name}", flush=True)

        except Exception as exc:
            msg = sanitize_message(
                exc,
                hidden_values=[real_file_name, record.get("filepath", "")],
            )
            print(
                f"\n[{i}/{len(records)}] SKIP {display_name}: "
                f"{type(exc).__name__}: {msg}",
                flush=True,
            )

            if PRINT_REAL_FILENAMES:
                traceback.print_exc()

        finally:
            if raw_before is not None:
                del raw_before

            gc.collect()

    print(f"Finished preprocessing loop. len(parts) = {len(parts)}", flush=True)

    if not parts:
        raise RuntimeError("No recordings were successfully processed.")

    if not apply_interpolation:
        parts, descriptions = keep_most_common_channel_count(
            parts,
            descriptions,
            dataset_label=str(records[0].get("origin", "dataset")) if records else "dataset",
        )

    return parts, descriptions


def make_dwt_combined_parts(
    parts: Sequence[mne.io.BaseRaw],
    descriptions: Sequence[dict[str, Any]],
    bands: Sequence[str] = DWT_COMBINED_BANDS,
    random_state: int = DWT_COMBINED_RANDOM_STATE,
) -> tuple[list[mne.io.BaseRaw], list[dict[str, Any]]]:
    """Create artificial DWT-combined recordings by mixing bands within each class."""
    rng = random.Random(random_state)
    by_label: dict[str, list[int]] = {}

    for idx, desc in enumerate(descriptions):
        by_label.setdefault(str(desc["label"]), []).append(idx)

    combined_parts: list[mne.io.BaseRaw] = []
    combined_descriptions: list[dict[str, Any]] = []

    for idx, (template_raw, desc) in enumerate(zip(parts, descriptions)):
        label = str(desc["label"])
        candidates = by_label[label]

        donor_indices = {band: rng.choice(candidates) for band in bands}
        donor_raws = {band: parts[donor_idx] for band, donor_idx in donor_indices.items()}

        print(f"  DWT combined {desc['file']}: {donor_indices}", flush=True)

        combined_raw = create_dwt_combined_raw(template_raw, donor_raws, bands=bands)
        set_background_annotation(combined_raw, label)

        new_desc = dict(desc)
        new_desc.update({
            "filter_type": "DWT_COMBINED",
            "frequency_band": "+".join(bands),
            "dwt_combined_bands": "+".join(bands),
            "dwt_combined_random_state": random_state,
            "dwt_donor_indices": str(donor_indices),
            "n_channels_after": len(combined_raw.ch_names),
            "channels_after": ",".join(combined_raw.ch_names),
        })

        combined_parts.append(combined_raw)
        combined_descriptions.append(new_desc)

    return combined_parts, combined_descriptions


def build_base_dataset(
    parts: Sequence[mne.io.BaseRaw],
    descriptions: Sequence[dict[str, Any]],
) -> BaseConcatDataset:
    """Build a Braindecode BaseConcatDataset from processed Raw parts."""
    return BaseConcatDataset([
        BaseDataset(raw, pd.Series(desc))
        for raw, desc in zip(parts, descriptions)
    ])


def create_windows_dataset(base_dataset: BaseConcatDataset) -> BaseConcatDataset:
    """Create fixed 4-second Braindecode windows from annotated recordings."""
    window_size_samples = int(WINDOW_DURATION_SEC * OUTPUT_SFREQ)

    return create_windows_from_events(
        base_dataset,
        trial_start_offset_samples=0,
        trial_stop_offset_samples=0,
        window_size_samples=window_size_samples,
        window_stride_samples=window_size_samples,
        drop_last_window=False,
        mapping={"Bgrnd": 0, "Bgrnd_autism": 1},
        n_jobs=4,
    )


def print_dataset_summary(
    base_dataset: BaseConcatDataset,
    windows_dataset: BaseConcatDataset,
) -> None:
    """Print the key dataset and windowing statistics."""
    n_classes = 2
    n_channels = windows_dataset[0][0].shape[0]
    input_window_samples = windows_dataset[0][0].shape[1]

    print(
        f"\nlen(windows_dataset) = {len(windows_dataset)}, "
        f"n_classes = {n_classes}, "
        f"n_channels = {n_channels}, "
        f"input_window_samples = {input_window_samples}",
        flush=True,
    )

    print("\nBase dataset description:", flush=True)
    print(base_dataset.description, flush=True)

    print("\nClass counts:", flush=True)
    print(base_dataset.description["label"].value_counts(), flush=True)


def save_base_and_windows(
    base_dataset: BaseConcatDataset,
    windows_dataset: BaseConcatDataset,
    dataset_dir: PathLike,
    windows_dataset_dir: PathLike,
    overwrite: bool = True,
) -> None:
    """Save the base and windowed Braindecode datasets."""
    print(f"Saving base dataset to: {dataset_dir}", flush=True)
    base_dataset.save(path=str(dataset_dir), overwrite=overwrite)

    print(f"Saving windows dataset to: {windows_dataset_dir}", flush=True)
    windows_dataset.save(path=str(windows_dataset_dir), overwrite=overwrite)


def load_saved_base_dataset(
    dataset_dir: PathLike,
    preload: bool = False,
) -> BaseConcatDataset:
    """Load a saved base Braindecode dataset from disk."""
    print(f"Loading saved base dataset: {dataset_dir}", flush=True)

    return load_concat_dataset(
        path=str(dataset_dir),
        preload=preload,
        ids_to_load=None,
        target_name=None,
        n_jobs=1,
    )


def source_output_names_for_mixed(
    filter_name: str,
    band: Optional[str],
) -> tuple[str, str]:
    """Return the Original and Sheffield output names needed for a Mixed dataset."""
    return (
        output_name_for("Original", filter_name, band=band, apply_interpolation=True),
        output_name_for("Sheffield", filter_name, band=band, apply_interpolation=True),
    )


def create_mixed_dataset(
    filter_name: str,
    band: Optional[str],
    overwrite: bool = True,
) -> dict[str, Any]:
    """Create a saved Mixed dataset by concatenating saved Original and Sheffield base datasets."""
    original_name, sheffield_name = source_output_names_for_mixed(filter_name, band)
    output_name = output_name_for("Mixed", filter_name, band=band, apply_interpolation=True)
    paths = dataset_paths(output_name)

    original_dataset = load_saved_base_dataset(dataset_paths(original_name)["dataset"])
    sheffield_dataset = load_saved_base_dataset(dataset_paths(sheffield_name)["dataset"])

    mixed_dataset = BaseConcatDataset(
        list(original_dataset.datasets) + list(sheffield_dataset.datasets)
    )

    print(f"mixed n_base_datasets: {len(mixed_dataset.datasets)}", flush=True)

    windows_dataset = create_windows_dataset(mixed_dataset)

    print_dataset_summary(mixed_dataset, windows_dataset)

    save_base_and_windows(
        mixed_dataset,
        windows_dataset,
        paths["dataset"],
        paths["windows"],
        overwrite=overwrite,
    )

    print("ALL DONE", flush=True)

    return {
        "origin": "Mixed",
        "filter_type": filter_name,
        "band": band,
        "dataset_dir": paths["dataset"],
        "windows_dataset_dir": paths["windows"],
        "n_base_datasets": len(mixed_dataset.datasets),
        "n_windows": len(windows_dataset),
    }


def create_dataset(
    origin: str,
    filter_type: Optional[str] = None,
    band: Optional[str] = None,
    filter_options: Optional[dict[str, Any]] = None,
    apply_interpolation: bool = True,
    overwrite: bool = False,
    save_plots: bool = True,
) -> dict[str, Any]:
    """Run the full preprocessing pipeline and save one dataset combination."""
    options = dict(filter_options or {})

    origin = normalize_origin(origin)
    filter_name = normalize_filter_type(filter_type)

    band = band or options.get("band") or options.get("frequency_band")
    if isinstance(band, str):
        band = band.lower()

    if filter_name == "DWT" and band is None:
        raise ValueError("DWT dataset creation requires a frequency band.")

    if filter_name != "DWT":
        band = None

    if origin == "Mixed":
        if not apply_interpolation:
            raise ValueError("Mixed no-interpolation dataset is unsupported by design.")

        return create_mixed_dataset(
            filter_name=filter_name,
            band=band,
            overwrite=overwrite,
        )

    output_name = output_name_for(
        origin,
        filter_name,
        band=band,
        apply_interpolation=apply_interpolation,
    )

    paths = dataset_paths(output_name)

    os.makedirs(paths["plots"], exist_ok=True)
    os.makedirs(paths["ica_plots"], exist_ok=True)

    records = load_records(origin)

    if origin == "Original":
        print("\nChecking Original recordings with background annotations...", flush=True)

        for record in records:
            matched = [
                key for key in BACKGROUND_KEYS
                if key in record.get("event_dict", {}).keys()
            ]
            print(
                f"{public_record_name(record)} | matched background annotations = {matched}",
                flush=True,
            )

        records = [record for record in records if record_has_background(record)]

    print(f"\nlen(records_filtered) = {len(records)}\n", flush=True)

    if not records:
        raise RuntimeError("No recordings matched the preprocessing criteria.")

    base_filter_name = "None" if filter_name == "DWT_COMBINED" else filter_name

    parts, descriptions = preprocess_records_to_parts(
        records=records,
        filter_name=base_filter_name,
        band=band,
        apply_interpolation=apply_interpolation,
        plots_dir=paths["plots"],
        ica_plots_dir=paths["ica_plots"],
        save_plots=save_plots,
        plot_first_n=int(options.get("plot_first_n", PLOT_FIRST_N)),
    )

    if filter_name == "DWT_COMBINED":
        parts, descriptions = make_dwt_combined_parts(
            parts,
            descriptions,
            bands=tuple(options.get("dwt_bands", DWT_COMBINED_BANDS)),
            random_state=int(options.get("random_state", DWT_COMBINED_RANDOM_STATE)),
        )

    base_dataset = build_base_dataset(parts, descriptions)
    windows_dataset = create_windows_dataset(base_dataset)

    print_dataset_summary(base_dataset, windows_dataset)

    save_base_and_windows(
        base_dataset,
        windows_dataset,
        paths["dataset"],
        paths["windows"],
        overwrite=overwrite,
    )

    print(f"\nSaved plots to: {paths['plots']}", flush=True)
    print("ALL DONE", flush=True)

    return {
        "origin": origin,
        "filter_type": filter_name,
        "band": band,
        "apply_interpolation": apply_interpolation,
        "dataset_dir": paths["dataset"],
        "windows_dataset_dir": paths["windows"],
        "plots_dir": paths["plots"],
        "n_base_datasets": len(base_dataset.datasets),
        "n_windows": len(windows_dataset),
    }

Preprocessing helpers loaded. Set RUN_DATASET_CREATION=True before running dataset cells.


## Implementation notes

The no-interpolation datasets no longer force the full target channel set. They crop each recording to 150 seconds, drop auxiliary/empty/non-finite channels, skip manually excluded subjects and recordings with zero usable EEG channels, and keep the recording-specific EEG channel set. Interpolated datasets still use the target 10--20 channel set. The DWT delta band keeps both the level-6 detail and approximation coefficients so the reconstruction better covers the low 0--4 Hz range. DWT combined datasets are generated by randomly drawing DWT bands from recordings within the same class, using a fixed random seed.


## Dataset creation calls

Set `RUN_DATASET_CREATION = True` before running the creation cells. Each cell below creates one saved dataset folder.

In [6]:
RUN_DATASET_CREATION = True
created_datasets: dict[str, dict[str, object]] = {}
print("Dataset creation is disabled. Set RUN_DATASET_CREATION = True to write folders.")


Dataset creation is disabled. Set RUN_DATASET_CREATION = True to write folders.


### Original / None

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["original_none"] = create_dataset("Original", None)
else:
    print("Skipped Original / None")


### Sheffield / None

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["sheffield_none"] = create_dataset("Sheffield", None)
else:
    print("Skipped Sheffield / None")


### Mixed / None

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["mixed_none"] = create_dataset("Mixed", None)
else:
    print("Skipped Mixed / None")


### Original / Butterworth

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["original_butterworth"] = create_dataset("Original", 'Butterworth')
else:
    print("Skipped Original / Butterworth")


### Sheffield / Butterworth

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["sheffield_butterworth"] = create_dataset("Sheffield", 'Butterworth')
else:
    print("Skipped Sheffield / Butterworth")


### Mixed / Butterworth

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["mixed_butterworth"] = create_dataset("Mixed", 'Butterworth')
else:
    print("Skipped Mixed / Butterworth")


### Original / ICA

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["original_ica"] = create_dataset("Original", 'ICA')
else:
    print("Skipped Original / ICA")


### Sheffield / ICA

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["sheffield_ica"] = create_dataset("Sheffield", 'ICA')
else:
    print("Skipped Sheffield / ICA")


### Mixed / ICA

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["mixed_ica"] = create_dataset("Mixed", 'ICA')
else:
    print("Skipped Mixed / ICA")


### Original / DWT alpha

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["original_dwt_alpha"] = create_dataset("Original", "DWT", band="alpha")
else:
    print("Skipped Original / DWT alpha")


### Sheffield / DWT alpha

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["sheffield_dwt_alpha"] = create_dataset("Sheffield", "DWT", band="alpha")
else:
    print("Skipped Sheffield / DWT alpha")


### Mixed / DWT alpha

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["mixed_dwt_alpha"] = create_dataset("Mixed", "DWT", band="alpha")
else:
    print("Skipped Mixed / DWT alpha")


### Original / DWT beta

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["original_dwt_beta"] = create_dataset("Original", "DWT", band="beta")
else:
    print("Skipped Original / DWT beta")


### Sheffield / DWT beta

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["sheffield_dwt_beta"] = create_dataset("Sheffield", "DWT", band="beta")
else:
    print("Skipped Sheffield / DWT beta")


### Mixed / DWT beta

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["mixed_dwt_beta"] = create_dataset("Mixed", "DWT", band="beta")
else:
    print("Skipped Mixed / DWT beta")


### Original / DWT theta

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["original_dwt_theta"] = create_dataset("Original", "DWT", band="theta")
else:
    print("Skipped Original / DWT theta")


### Sheffield / DWT theta

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["sheffield_dwt_theta"] = create_dataset("Sheffield", "DWT", band="theta")
else:
    print("Skipped Sheffield / DWT theta")


### Mixed / DWT theta

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["mixed_dwt_theta"] = create_dataset("Mixed", "DWT", band="theta")
else:
    print("Skipped Mixed / DWT theta")


### Original / DWT gamma

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["original_dwt_gamma"] = create_dataset("Original", "DWT", band="gamma")
else:
    print("Skipped Original / DWT gamma")


### Sheffield / DWT gamma

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["sheffield_dwt_gamma"] = create_dataset("Sheffield", "DWT", band="gamma")
else:
    print("Skipped Sheffield / DWT gamma")


### Mixed / DWT gamma

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["mixed_dwt_gamma"] = create_dataset("Mixed", "DWT", band="gamma")
else:
    print("Skipped Mixed / DWT gamma")


### Original / DWT delta

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["original_dwt_delta"] = create_dataset("Original", "DWT", band="delta")
else:
    print("Skipped Original / DWT delta")


### Sheffield / DWT delta

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["sheffield_dwt_delta"] = create_dataset("Sheffield", "DWT", band="delta")
else:
    print("Skipped Sheffield / DWT delta")


### Mixed / DWT delta

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["mixed_dwt_delta"] = create_dataset("Mixed", "DWT", band="delta")
else:
    print("Skipped Mixed / DWT delta")


### Original / no interpolation

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["original_no_interpolation"] = create_dataset("Original", None, apply_interpolation=False)
else:
    print("Skipped Original / no interpolation")


### Sheffield / no interpolation

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["sheffield_no_interpolation"] = create_dataset("Sheffield", None, apply_interpolation=False)
else:
    print("Skipped Sheffield / no interpolation")


### Original / DWT combined bands

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["original_dwt_combined"] = create_dataset("Original", "DWT_COMBINED")
else:
    print("Skipped Original / DWT combined bands")


### Sheffield / DWT combined bands

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["sheffield_dwt_combined"] = create_dataset("Sheffield", "DWT_COMBINED")
else:
    print("Skipped Sheffield / DWT combined bands")


### Mixed / DWT combined bands

In [ ]:
if RUN_DATASET_CREATION:
    created_datasets["mixed_dwt_combined"] = create_dataset("Mixed", "DWT_COMBINED")
else:
    print("Skipped Mixed / DWT combined bands")
